# Notebook 9 — Geofencing and Regional Access Policies with IP2Location.io

**Use case:** Estimate whether a visitor should be shown a resource or promotion in a market using IP geolocation and configurable policies. The output is **ALLOW**, **DENY**, or **REVIEW** with an auditable reason.

This notebook shows a **decision-support prototype**, not a production authorization system or definitive proof of a person's residence. IP geolocation is not GPS: VPNs, mobile networks, corporate gateways, and shared IPs can point to a location other than the visitor's physical location. Production legal/geographic compliance must rely on additional verification and legal review.

**Inputs:** `sample_geofencing_visitors.csv`, `sample_geofencing_policies.csv`  
**Outputs:** `outputs_notebook_09/geofencing_visitor_decisions.csv`, `geofencing_policy_summary.csv`, `geofencing_review_queue.csv`, and `geofencing_reason_summary.csv`.

**Two modes:** `MODE = "demo"` runs entirely offline with **synthetic, deliberately fictional** IP geolocations. `MODE = "live"` calls the standard IP2Location.io API for unique globally routable IP addresses. All demo-to-IP associations are fictional and must not be interpreted as actual locations of those IPs.

## 1. Setup

Install missing packages in the active Jupyter kernel:

```python
%pip install pandas requests matplotlib
```

Run the cells **in order** after selecting `MODE`. The sample CSVs are supplied beside this notebook; if either is missing, the setup cell gives a clear error. API access is required only in live mode.

API reference: https://www.ip2location.io/ip2location-documentation

In [ ]:
from pathlib import Path
from collections import Counter
import ipaddress
import os
import re

import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

print('Libraries loaded.')

## 2. Shared configuration

Every variable used by live mode is initialized outside demo-only cells. Optional authentication uses the `IP2LOCATION_API_KEY` environment variable; never hard-code production secrets in the notebook.

**Decision order:** bad/unavailable geolocation → REVIEW; blocked country → DENY; country/region restrictions → DENY or REVIEW; proxy review → REVIEW; otherwise ALLOW. There is no automatic enforcement or network blocking.

In [ ]:
MODE = 'demo'  # Choose 'demo' or 'live'.
API_KEY = os.getenv('IP2LOCATION_API_KEY', '').strip()
API_URL = 'https://api.ip2location.io/'
INPUT_VISITORS = Path('sample_geofencing_visitors.csv')
INPUT_POLICIES = Path('sample_geofencing_policies.csv')
OUTPUT_DIR = Path('outputs_notebook_09')
MAX_UNIQUE_IP_LOOKUPS = 25  # Hard cap on distinct API queries per notebook run.
REQUEST_TIMEOUT_SECONDS = 15

VISITOR_COLUMNS = ['visitor_id', 'visitor_ip', 'resource_id', 'scenario_note']
POLICY_COLUMNS = [
    'resource_id', 'resource_label', 'allowed_countries',
    'blocked_countries', 'allowed_regions', 'proxy_handling', 'policy_note'
]
GEO_COLUMNS = [
    'country_code', 'country_name', 'region_name', 'city_name',
    'is_proxy', 'proxy_signal_available', 'lookup_status', 'lookup_error'
]
DECISION_COLUMNS = ['decision', 'decision_reason', 'policy_label']

assert MODE in {'demo', 'live'}
assert MAX_UNIQUE_IP_LOOKUPS >= 1
print(f'Mode: {MODE}; maximum unique live lookups: {MAX_UNIQUE_IP_LOOKUPS}')

## 3. Load sample visitors and policy rules

Policy CSV rules use `|` to separate countries (ISO 3166-1 alpha-2) and `COUNTRY:region_name` pairs for region restrictions. Blank allowlists mean *no allowlist restriction*. The `proxy_handling` column is either `ignore` or `review`.

These illustrative commercial restrictions do **not** represent real legal or licensing requirements.

In [ ]:
for path in (INPUT_VISITORS, INPUT_POLICIES):
    if not path.is_file():
        raise FileNotFoundError(f'{path} is missing. Download the sample CSV and place it alongside the notebook.')

visitors = pd.read_csv(INPUT_VISITORS, dtype=str, keep_default_na=False)
policies = pd.read_csv(INPUT_POLICIES, dtype=str, keep_default_na=False)
if not set(VISITOR_COLUMNS).issubset(visitors):
    raise ValueError('Visitor input is missing required columns.')
if not set(POLICY_COLUMNS).issubset(policies):
    raise ValueError('Policy input is missing required columns.')
visitors = visitors[VISITOR_COLUMNS].copy()
policies = policies[POLICY_COLUMNS].copy()
for col in ('visitor_id', 'resource_id'):
    if visitors[col].eq('').any():
        raise ValueError(f'Visitor {col} cannot be blank.')
if visitors['visitor_id'].duplicated().any():
    raise ValueError('visitor_id must be unique.')
if policies['resource_id'].eq('').any() or policies['resource_id'].duplicated().any():
    raise ValueError('Policy resource_id must be unique and nonempty.')

display(visitors.head(8))
display(policies)
print(f'Loaded {len(visitors)} visitor requests and {len(policies)} policies.')

## 4. Validate policies before evaluating visitors

Fail clearly on ambiguous or malformed rules. A country on both the allowlist and blocklist is rejected rather than silently picking a side. Matching region names is case-insensitive and requires an exact country/region pair; it is **not** GPS-level geofencing.

In [ ]:
def split_codes(value):
    values = [part.strip().upper() for part in str(value).split('|') if part.strip()]
    if any(not re.fullmatch(r'[A-Z]{2}', item) for item in values):
        raise ValueError(f'Invalid country code list: {value!r}')
    return set(values)

def split_region_rules(value):
    output = set()
    for item in str(value).split('|'):
        item = item.strip()
        if not item:
            continue
        if ':' not in item:
            raise ValueError(f'Invalid region rule {item!r}; use COUNTRY:region_name')
        country, region = (part.strip() for part in item.split(':', 1))
        if not re.fullmatch(r'[A-Za-z]{2}', country) or not region:
            raise ValueError(f'Invalid region rule {item!r}')
        output.add((country.upper(), region.casefold()))
    return output

policy_map = {}
for row in policies.to_dict('records'):
    allowed = split_codes(row['allowed_countries'])
    blocked = split_codes(row['blocked_countries'])
    regions = split_region_rules(row['allowed_regions'])
    if allowed & blocked:
        raise ValueError(f'Overlapping allow/deny countries for {row["resource_id"]}')
    if regions and (not allowed or any(code not in allowed for code, _ in regions)):
        raise ValueError(f'Region rules must belong to explicitly allowed countries: {row["resource_id"]}')
    handling = row['proxy_handling'].strip().lower()
    if handling not in {'ignore', 'review'}:
        raise ValueError(f'Unsupported proxy_handling for {row["resource_id"]}: {handling!r}')
    policy_map[row['resource_id']] = {
        'label': row['resource_label'], 'allow': allowed,
        'block': blocked, 'regions': regions, 'proxy_handling': handling
    }
print('Policy validation passed.')

## 5. Validate IPs and build the unique lookup queue

Only globally routable IPv4/IPv6 addresses are eligible for requests. Invalid and private/reserved addresses are never sent to the API. Each unique eligible IP is looked up once even if it requests several resources.

In [ ]:
def ip_status(value):
    try:
        addr = ipaddress.ip_address(str(value).strip())
    except ValueError:
        return 'invalid_ip'
    return 'public_ip' if addr.is_global else 'non_public_ip'

visitors['ip_status'] = visitors['visitor_ip'].map(ip_status)
unique_public_ips = list(dict.fromkeys(
    visitors.loc[visitors['ip_status'].eq('public_ip'), 'visitor_ip'].str.strip()
))
print('IP status:')
print(visitors['ip_status'].value_counts().to_string())
print(f'Unique eligible public IPs: {len(unique_public_ips)}')

## 6. IP2Location.io client and response normalization

The basic response includes `country_code`, `region_name`, and `is_proxy`, but **proxy coverage varies by subscription**. Treat `is_proxy = false` only as a negative result for the detection scope available to that API plan, not proof that an IP is safe. A missing flag means unknown — never silently assume `False`.

In demo mode, responses are synthetic fixtures coded directly in this cell; live mode does not need to run any separate demo-only cells. The API uses HTTPS GET with optional Bearer token and handles documented API error objects (`error.error_code`) as well as HTTP failures.

In [ ]:
def normalized_proxy(value):
    if isinstance(value, bool):
        return value
    if value in (0, 1):
        return bool(value)
    if isinstance(value, str):
        val = value.strip().lower()
        if val in {'1', 'true', 'yes'}:
            return True
        if val in {'0', 'false', 'no'}:
            return False
    return None

def geo_error(status, message=''):
    return dict(country_code='', country_name='', region_name='', city_name='',
                is_proxy=None, proxy_signal_available=False,
                lookup_status=status, lookup_error=str(message))

def parse_geolocation(payload):
    if not isinstance(payload, dict):
        return geo_error('api_error', 'Unexpected response format')
    if 'error' in payload:
        detail = payload.get('error') or {}
        code = detail.get('error_code', '') if isinstance(detail, dict) else ''
        msg = detail.get('error_message', 'API error') if isinstance(detail, dict) else str(detail)
        status = 'quota_or_auth_error' if str(code) == '10000' else 'api_error'
        return geo_error(status, f'{code}: {msg}')
    cc = str(payload.get('country_code') or '').strip().upper()
    if not re.fullmatch(r'[A-Z]{2}', cc):
        return geo_error('incomplete_location', 'Missing valid two-letter country_code')
    p = normalized_proxy(payload.get('is_proxy'))
    return {
        'country_code': cc, 'country_name': str(payload.get('country_name') or ''),
        'region_name': str(payload.get('region_name') or ''),
        'city_name': str(payload.get('city_name') or ''),
        'is_proxy': p, 'proxy_signal_available': p is not None,
        'lookup_status': 'ok', 'lookup_error': ''
    }

# These are fictional demo assignments, NOT real locations of the IP addresses.
# A special marker simulates one failed lookup.
DEMO_GEO = {
    '8.8.8.8': ('US', 'United States', 'California', 'Demo City', False),
    '1.1.1.1': ('AU', 'Australia', 'Queensland', 'Demo City', False),
    '9.9.9.9': ('DE', 'Germany', 'Berlin', 'Demo City', False),
    '208.67.222.222': ('US', 'United States', 'Texas', 'Demo City', True),
    '8.26.56.26': ('MY', 'Malaysia', 'Kuala Lumpur', 'Demo City', False),
    '76.76.2.0': ('SG', 'Singapore', 'Singapore', 'Demo City', False),
    '149.112.112.112': ('FR', 'France', 'Île-de-France', 'Demo City', False),
    '114.114.114.114': ('CN', 'China', 'Jiangsu', 'Demo City', False),
    '94.140.14.14': ('GB', 'United Kingdom', 'England', 'Demo City', True),
    '4.2.2.2': ('US', 'United States', '', 'Demo City', False),
    '2001:4860:4860::8888': ('US', 'United States', 'California', 'Demo City', False),
    '64.6.64.6': ('JP', 'Japan', 'Tokyo', 'Demo City', False),
    '185.228.168.9': ('GB', 'United Kingdom', 'England', 'Demo City', None),
}

def demo_lookup(ip):
    if ip == '45.90.28.0':
        return geo_error('api_error', 'Synthetic upstream lookup failure')
    item = DEMO_GEO.get(ip)
    if item is None:
        return geo_error('demo_fixture_missing', 'No synthetic location for this IP')
    code, country, region, city, is_proxy = item
    payload = {'country_code': code, 'country_name': country,
               'region_name': region, 'city_name': city}
    if is_proxy is not None:
        payload['is_proxy'] = is_proxy
    return parse_geolocation(payload)

SESSION = requests.Session()
if API_KEY:
    SESSION.headers.update({'Authorization': f'Bearer {API_KEY}'})

def live_lookup(ip):
    try:
        response = SESSION.get(API_URL, params={'ip': ip, 'format': 'json'},
                               timeout=REQUEST_TIMEOUT_SECONDS)
    except requests.RequestException as exc:
        return geo_error('network_error', type(exc).__name__ + ': ' + str(exc))
    if response.status_code == 429:
        return geo_error('quota_or_auth_error', 'HTTP 429: rate limit reached')
    try:
        payload = response.json()
    except ValueError:
        return geo_error('api_error', f'HTTP {response.status_code}: invalid JSON')
    parsed = parse_geolocation(payload)
    if response.status_code in (401, 402, 403):
        return geo_error('quota_or_auth_error', f'HTTP {response.status_code}: {parsed["lookup_error"]}')
    if response.status_code >= 400 and parsed['lookup_status'] == 'ok':
        return geo_error('api_error', f'HTTP {response.status_code}')
    return parsed

print(f'Lookup implementation ready for {MODE} mode.')

## 7. Run deduplicated lookups

The `MAX_UNIQUE_IP_LOOKUPS` cap protects your quota. If a quota/authentication error is returned, further live lookups are stopped. Rows that were not checked remain `REVIEW` later. **No API key or access token is written to the CSV exports.**

In [ ]:
lookup_cache = {}
stop_new_requests = False
for index, ip in enumerate(unique_public_ips):
    if MODE == 'live' and index >= MAX_UNIQUE_IP_LOOKUPS:
        lookup_cache[ip] = geo_error('not_queried_budget', 'Local unique-IP budget reached')
        continue
    if MODE == 'live' and stop_new_requests:
        lookup_cache[ip] = geo_error('not_queried_after_quota', 'Stopped after quota/authentication error')
        continue
    result = demo_lookup(ip) if MODE == 'demo' else live_lookup(ip)
    lookup_cache[ip] = result
    if MODE == 'live' and result['lookup_status'] == 'quota_or_auth_error':
        stop_new_requests = True

print('Lookup status by unique public IP:')
print(pd.Series([v['lookup_status'] for v in lookup_cache.values()]).value_counts().to_string())

In [ ]:
def geo_for_row(row):
    if row.ip_status != 'public_ip':
        return geo_error(row.ip_status, 'Address was not sent to API')
    return lookup_cache[row.visitor_ip.strip()]

enriched_rows = []
for row in visitors.itertuples(index=False):
    item = row._asdict()
    item.update(geo_for_row(row))
    enriched_rows.append(item)
enriched = pd.DataFrame(enriched_rows)
assert len(enriched) == len(visitors)
display(enriched[['visitor_id','visitor_ip','resource_id','country_code','region_name',
                  'is_proxy','lookup_status']].head(12))

## 8. Apply each policy with explicit Allow / Deny / Review decisions

The policy is evaluated per **visitor × requested resource**, not per IP alone. Country restrictions are applied before optional proxy review. Missing fields cannot produce a confident location-based ALLOW. An unknown resource or invalid request is REVIEW, not a default allow.

For a real paywall, legal restriction, or high-stakes service, use a trusted account/entitlement check and independently verified residence or license information. This notebook alone must not be the enforcement point.

In [ ]:
def evaluate_policy(row):
    policy = policy_map.get(row['resource_id'])
    if policy is None:
        return 'REVIEW', 'unknown_resource', ''
    label = policy['label']
    if row['lookup_status'] != 'ok':
        return 'REVIEW', 'location_unavailable:' + str(row['lookup_status']), label
    country = str(row['country_code']).upper()
    region = str(row['region_name']).strip().casefold()

    if country in policy['block']:
        return 'DENY', 'blocked_country', label
    if policy['allow'] and country not in policy['allow']:
        return 'DENY', 'outside_allowed_countries', label
    if policy['regions']:
        if not region:
            return 'REVIEW', 'region_missing', label
        if (country, region) not in policy['regions']:
            return 'DENY', 'outside_allowed_regions', label
    if policy['proxy_handling'] == 'review':
        if row['is_proxy'] is True:
            return 'REVIEW', 'proxy_detected', label
        if row['is_proxy'] is not False:
            return 'REVIEW', 'proxy_signal_unavailable', label
    return 'ALLOW', 'eligible_under_example_policy', label

outcomes = [evaluate_policy(r) for r in enriched.to_dict('records')]
decisions = enriched.copy()
decisions[DECISION_COLUMNS] = pd.DataFrame(outcomes, columns=DECISION_COLUMNS,
                                             index=decisions.index)
display(decisions[['visitor_id','resource_id','country_code','region_name','lookup_status',
                   'is_proxy','decision','decision_reason']])

## 9. Summarize by policy and decision reason

Use the review queue to resolve missing location, limited subscription coverage, or a policy mismatch with an independent source. Do not interpret DENY as a legal/geographic certainty.

In [ ]:
policy_summary = (decisions.groupby(['resource_id','decision']).size().unstack(fill_value=0)
                  .reindex(columns=['ALLOW', 'DENY', 'REVIEW'], fill_value=0)
                  .reset_index())
policy_summary['total'] = policy_summary[['ALLOW','DENY','REVIEW']].sum(axis=1)
reason_summary = (decisions.groupby(['decision','decision_reason'], dropna=False)
                  .size().reset_index(name='requests')
                  .sort_values(['decision','requests'], ascending=[True,False]))
review_queue = decisions.loc[decisions['decision'].eq('REVIEW')].copy()
print('Decision counts:')
print(decisions['decision'].value_counts().to_string())
display(policy_summary)
display(reason_summary)

## 10. Visualize policy decisions

These counts reflect the supplied example requests; a different input file or real IP geolocation response will produce different results.

In [ ]:
counts = decisions['decision'].value_counts().reindex(['ALLOW','DENY','REVIEW'], fill_value=0)
fig, ax = plt.subplots(figsize=(7.6, 4))
counts.plot(kind='bar', ax=ax)
ax.set_title('Geofencing decisions')
ax.set_xlabel('Policy decision')
ax.set_ylabel('Number of visitor requests')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
plt.show()

In [ ]:
policy_chart = policy_summary.set_index('resource_id')[['ALLOW','DENY','REVIEW']]
fig, ax = plt.subplots(figsize=(9, 4.4))
policy_chart.plot(kind='barh', stacked=True, ax=ax)
ax.set_title('Decisions by requested resource')
ax.set_xlabel('Visitor requests')
ax.set_ylabel('Resource')
fig.tight_layout()
plt.show()

## 11. Export decisions and review reports

CSV files contain visitor IP addresses. Store and share them under the same appropriate controls as other website access data. Prefer short retention and delete exports that are no longer required.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    'geofencing_visitor_decisions.csv': decisions,
    'geofencing_policy_summary.csv': policy_summary,
    'geofencing_review_queue.csv': review_queue,
    'geofencing_reason_summary.csv': reason_summary,
}
for filename, frame in exports.items():
    destination = OUTPUT_DIR / filename
    frame.to_csv(destination, index=False, encoding='utf-8-sig')
    print(f'Saved {destination} ({len(frame)} rows)')

## 12. Self-checks and practical next steps

- **IP geolocation is approximate.** It may identify a network exit point rather than the visitor's real location.
- **Policy data is illustrative.** Never treat the example country rules as actual legal or licensing guidance.
- **Proxy data is plan-dependent.** `is_proxy = False` is not a guarantee; absence of the field is unknown.
- **Do not rely on IP alone to enforce a legal or high-stakes restriction.** Combine server-side policy checks, account entitlements and independently verified location where necessary.
- **API efficiency:** only unique globally routable IPs are queried, and the request budget is configurable.
- **Production:** run server-side, protect credentials, add caching with suitable retention, monitoring, rate limits, independent overrides/appeals, and legal review.

API documentation: https://www.ip2location.io/ip2location-documentation  
Geolocation limitations: https://www.ip2location.io/solutions/localization

In [ ]:
assert len(decisions) == len(visitors)
assert decisions['visitor_id'].is_unique
assert set(decisions['decision']).issubset({'ALLOW','DENY','REVIEW'})
assert int(policy_summary[['ALLOW','DENY','REVIEW']].to_numpy().sum()) == len(visitors)
assert len(review_queue) == int(decisions['decision'].eq('REVIEW').sum())
assert decisions.loc[decisions['lookup_status'] != 'ok', 'decision'].eq('REVIEW').all()
assert decisions.loc[decisions['resource_id'] == 'unknown_resource', 'decision'].eq('REVIEW').all()
assert decisions.loc[decisions['visitor_id'] == 'V016', 'decision'].iloc[0] == 'REVIEW'
assert decisions.loc[decisions['visitor_id'] == 'V017', 'decision'].iloc[0] == 'REVIEW'
print('Self-checks passed: row totals, decision integrity, and safe fallback behavior.')